# TMDB Batch Data Ingestion

This notebook handles large-scale ingestion of Movie and TV Show metadata
from TMDB Details API into PostgreSQL Bronze layer.

## Ingestion Strategy
- Stream TMDB Daily ID Exports
- Process IDs in batches
- Retrieve metadata through TMDB Details API
- Preserve raw JSON responses
- Prevent duplicate records
- Track ingestion progress through checkpoints

# Libraries

In [57]:
import importlib
import common

importlib.reload(common)

from common import *

engine = get_engine()


In [58]:
with engine.connect() as conn:
    result = conn.execute(text("""
        SELECT schema_name
        FROM information_schema.schemata
        ORDER BY schema_name;
    """))

    schemas = result.fetchall()

    for schema in schemas:
        print(schema[0])

bronze
gold
information_schema
ops
pg_catalog
pg_toast
public
silver


# Taking datas from apis

In [59]:
base_url ="https://api.themoviedb.org/3"

movie_export_url = "https://files.tmdb.org/p/exports/movie_ids_{date}.json.gz"
tv_export_url = "https://files.tmdb.org/p/exports/tv_series_ids_{date}.json.gz"

In [60]:
batch_size = 500
request_timeout = 30

print(f'TMBD token: {'confirmed' if TMDB_API_TOKEN else 'missing'}')
print(f'Batch size: {batch_size}')

TMBD token: confirmed
Batch size: 500


In [61]:
export_date = '11_19_2026'

movie_export_url = movie_export_url.format(date=export_date)
tv_export_url = tv_export_url.format(date=export_date)

In [62]:
import gzip

def load_export_ids(url):
    ids = set()
    total = 0
    missing = 0
    duplicates = 0

    with requests.get(url, stream=True, timeout=request_timeout) as response:
        response.raise_for_status()
        response.raw.decode_content = False

        with gzip.GzipFile(fileobj=response.raw) as file:
            for line in file:
                record = json.loads(line)
                tmdb_id = record.get("id")

                total += 1

                if tmdb_id is None:
                    missing += 1
                elif tmdb_id in ids:
                    duplicates += 1
                else:
                    ids.add(tmdb_id)

    return ids, total, missing, duplicates

In [63]:
movie_ids, movie_total, movie_missing, movie_duplicate = load_export_ids(movie_export_url)

tv_ids, tv_total, tv_missing, tv_duplicates = load_export_ids(tv_export_url)

In [64]:
display('MOVIES')
print(f'unique IDs: {len(movie_ids)}')
print(f'Total records: {movie_total}')
print(f'Missing IDs: {movie_missing}')
print(f'Duplicates: {movie_duplicate}')

print('\nTV SHOWS')
print(f'Unique IDs: {len(tv_ids)}')
print(f'Total records: {tv_total}')
print(f'Missing tv shows: {tv_missing}')
print(f'Duplicates: {tv_duplicates}')

'MOVIES'

unique IDs: 1278958
Total records: 1278958
Missing IDs: 0
Duplicates: 0

TV SHOWS
Unique IDs: 237134
Total records: 237134
Missing tv shows: 0
Duplicates: 0


In [65]:
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS bronze.tmdb_metadata (
            media_type VARCHAR(10) NOT NULL,
            tmdb_id BIGINT NOT NULL,
            payload JSONB NOT NULL,
            ingested_at TIMESTAMPTZ DEFAULT NOW(),
            PRIMARY KEY (media_type, tmdb_id)
        );
    """))

In [66]:
with engine.connect() as conn:
    existing_movie_ids = set(
        conn.execute(text('''
            select tmdb_id
            from bronze.tmdb_metadata
            where media_type = 'movie'   ''')).scalars().all()
    )

    existing_tv_ids = set(
        conn.execute(text('''
            select tmdb_id
            from bronze.tmdb_metadata
            where media_type = 'tv' ''')).scalars().all()
    )

In [67]:
print(f'Existing moves: {len(existing_movie_ids)}')
print(f'Existing tv shows: {len(existing_movie_ids)}')

Existing moves: 0
Existing tv shows: 0


In [68]:
new_movie_ids = movie_ids - existing_movie_ids
new_tv_ids = tv_ids - existing_tv_ids

In [69]:
print(f"New Movies: {len(new_movie_ids)}")
print(f"New TV Shows: {len(new_tv_ids)}")
print(f"Total New IDs: {len(new_movie_ids) + len(new_tv_ids)}")

New Movies: 1278958
New TV Shows: 237134
Total New IDs: 1516092


In [70]:
print("Movie IDs:", len(movie_ids))
print("TV IDs:", len(tv_ids))

print("Existing Movie IDs:", len(existing_movie_ids))
print("Existing TV IDs:", len(existing_tv_ids))

print("New Movie IDs:", len(new_movie_ids))
print("New TV IDs:", len(new_tv_ids))

Movie IDs: 1278958
TV IDs: 237134
Existing Movie IDs: 0
Existing TV IDs: 0
New Movie IDs: 1278958
New TV IDs: 237134


In [71]:
def get_details(tmdb_id, media_type):
    url = f"{BASE_URL}/{media_type}/{tmdb_id}"

    headers = {
        "Authorization": f"Bearer {TMDB_API_TOKEN}"
    }

    response = requests.get(
        url,
        headers=headers,
        timeout=REQUEST_TIMEOUT
    )

    if response.status_code == 200:
        return response.json()

    return None